In [6]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=0.8, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0) #+ cols_v1)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_2d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).values
    Y = df[TARGET].fillna(0).astype(np.int32).values
    return X, Y

Xtr, Ytr = prepare_data_2d(train_df, feat_cols)
Xva, Yva = prepare_data_2d(val_df, feat_cols)
Xte, Yte = prepare_data_2d(test_df, feat_cols)

print("=== Data Shapes for XGBoost ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")

# =========================
# MODEL BUILD & TRAINING (Fixed Params)
# =========================
print("\n[Final] Retraining XGBoost model on full Train set with best hyperparameters...")
best_params_xgb = {
      "n_estimators": 450,
      "learning_rate": 0.0608598829841135,
      "max_depth": 3,
      "min_child_weight": 4,
      "subsample": 0.5733779454085566,
      "colsample_bytree": 0.5461692973843989,
      "gamma": 0.9313010568883545,
      "reg_alpha": 0.0011840345146135153,
      "reg_lambda": 0.00240217612024316,
      "objective": "binary:logistic",
      "eval_metric": "auc",
      "booster": "gbtree",
      "tree_method": "hist",
      "device": "cuda",
      "random_state": 1,
      "n_jobs": -1
    }

best_params_xgb.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
})

# 최종 모델은 전체 Xtr로 학습, Xva를 Early Stopping 검증용으로 사용
final_model = xgb.XGBClassifier(**best_params_xgb, early_stopping_rounds=50)

final_model.fit(
    Xtr, Ytr,
    eval_set=[(Xva, Yva)],
    verbose=False
)

# =========================
# FINAL EVALUATION
# =========================
y_prob = final_model.predict_proba(Xte)[:, 1]
y_true = Yte

# [수정] 사용된 피처 개수 추출 (XGBoost 입력 데이터의 컬럼 수)
n_features_used = Xte.shape[1]

# [수정] n_features 인자 추가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, 
    thr=OPERATING_THR,
    n_features=n_features_used  # ★ 이 부분을 추가
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    # "best_params_xgb": best_params_xgb  # (옵션) 파라미터 저장
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)


[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...
   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...
   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...
   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...
   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)
[dataset(raw)] n=579,551  pos=43,352  rate=0.074803
[dataset(after 100K sampling)] n=100,000  pos=10,000  rate=0.100000
[check dataset(100K)] pos_rate=0.100000  target=0.10  diff=+0.000000  OK
=== Data Shapes for XGBoost ===
Train: X=(70000, 41), Y=(70000,)
Val  : X=(15000, 41), Y=(15000,)
Test : X=(15000, 41), Y=(15000,)

[Final] Retraining XGBoost model on full Train set with best hyperparameters

In [1]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=0.8, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0 + cols_v1)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_2d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).values
    Y = df[TARGET].fillna(0).astype(np.int32).values
    return X, Y

Xtr, Ytr = prepare_data_2d(train_df, feat_cols)
Xva, Yva = prepare_data_2d(val_df, feat_cols)
Xte, Yte = prepare_data_2d(test_df, feat_cols)

print("=== Data Shapes for XGBoost ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")

# =========================
# MODEL BUILD & TRAINING (Fixed Params)
# =========================
print("\n[Final] Retraining XGBoost model on full Train set with best hyperparameters...")
best_params_xgb = {
      "n_estimators": 450,
      "learning_rate": 0.0608598829841135,
      "max_depth": 3,
      "min_child_weight": 4,
      "subsample": 0.5733779454085566,
      "colsample_bytree": 0.5461692973843989,
      "gamma": 0.9313010568883545,
      "reg_alpha": 0.0011840345146135153,
      "reg_lambda": 0.00240217612024316,
      "objective": "binary:logistic",
      "eval_metric": "auc",
      "booster": "gbtree",
      "tree_method": "hist",
      "device": "cuda",
      "random_state": 1,
      "n_jobs": -1
    }

best_params_xgb.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
})

# 최종 모델은 전체 Xtr로 학습, Xva를 Early Stopping 검증용으로 사용
final_model = xgb.XGBClassifier(**best_params_xgb, early_stopping_rounds=50)

final_model.fit(
    Xtr, Ytr,
    eval_set=[(Xva, Yva)],
    verbose=False
)

# =========================
# FINAL EVALUATION
# =========================
y_prob = final_model.predict_proba(Xte)[:, 1]
y_true = Yte

# [수정] 사용된 피처 개수 추출 (XGBoost 입력 데이터의 컬럼 수)
n_features_used = Xte.shape[1]

# [수정] n_features 인자 추가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, 
    thr=OPERATING_THR,
    n_features=n_features_used  # ★ 이 부분을 추가
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    # "best_params_xgb": best_params_xgb  # (옵션) 파라미터 저장
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)


2025-12-17 00:12:52.213197: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 00:12:52.222730: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765897972.233296 3385091 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765897972.236486 3385091 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765897972.245105 3385091 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...
   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...
   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...
   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...
   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)
[dataset(raw)] n=579,551  pos=43,352  rate=0.074803
[dataset(after 100K sampling)] n=100,000  pos=10,000  rate=0.100000
[check dataset(100K)] pos_rate=0.100000  target=0.10  diff=+0.000000  OK
=== Data Shapes for XGBoost ===
Train: X=(70000, 44), Y=(70000,)
Val  : X=(15000, 44), Y=(15000,)
Test : X=(15000, 44), Y=(15000,)

[Final] Retraining XGBoost model on full Train set with best hyperparameters

In [2]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=0.8, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0 + cols_v2)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_2d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).values
    Y = df[TARGET].fillna(0).astype(np.int32).values
    return X, Y

Xtr, Ytr = prepare_data_2d(train_df, feat_cols)
Xva, Yva = prepare_data_2d(val_df, feat_cols)
Xte, Yte = prepare_data_2d(test_df, feat_cols)

print("=== Data Shapes for XGBoost ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")

# =========================
# MODEL BUILD & TRAINING (Fixed Params)
# =========================
print("\n[Final] Retraining XGBoost model on full Train set with best hyperparameters...")
best_params_xgb = {
      "n_estimators": 450,
      "learning_rate": 0.0608598829841135,
      "max_depth": 3,
      "min_child_weight": 4,
      "subsample": 0.5733779454085566,
      "colsample_bytree": 0.5461692973843989,
      "gamma": 0.9313010568883545,
      "reg_alpha": 0.0011840345146135153,
      "reg_lambda": 0.00240217612024316,
      "objective": "binary:logistic",
      "eval_metric": "auc",
      "booster": "gbtree",
      "tree_method": "hist",
      "device": "cuda",
      "random_state": 1,
      "n_jobs": -1
    }

best_params_xgb.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
})

# 최종 모델은 전체 Xtr로 학습, Xva를 Early Stopping 검증용으로 사용
final_model = xgb.XGBClassifier(**best_params_xgb, early_stopping_rounds=50)

final_model.fit(
    Xtr, Ytr,
    eval_set=[(Xva, Yva)],
    verbose=False
)

# =========================
# FINAL EVALUATION
# =========================
y_prob = final_model.predict_proba(Xte)[:, 1]
y_true = Yte

# [수정] 사용된 피처 개수 추출 (XGBoost 입력 데이터의 컬럼 수)
n_features_used = Xte.shape[1]

# [수정] n_features 인자 추가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, 
    thr=OPERATING_THR,
    n_features=n_features_used  # ★ 이 부분을 추가
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    # "best_params_xgb": best_params_xgb  # (옵션) 파라미터 저장
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)


[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...
   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...
   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...
   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...
   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)
[dataset(raw)] n=579,551  pos=43,352  rate=0.074803
[dataset(after 100K sampling)] n=100,000  pos=10,000  rate=0.100000
[check dataset(100K)] pos_rate=0.100000  target=0.10  diff=+0.000000  OK
=== Data Shapes for XGBoost ===
Train: X=(70000, 47), Y=(70000,)
Val  : X=(15000, 47), Y=(15000,)
Test : X=(15000, 47), Y=(15000,)

[Final] Retraining XGBoost model on full Train set with best hyperparameters

In [1]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)
path_v0 = os.path.join(DATA_DIR, "featured_all.parquet")
final_data = pd.read_parquet(path_v0)

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=0.8, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0 + cols_v3)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_2d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).values
    Y = df[TARGET].fillna(0).astype(np.int32).values
    return X, Y

Xtr, Ytr = prepare_data_2d(train_df, feat_cols)
Xva, Yva = prepare_data_2d(val_df, feat_cols)
Xte, Yte = prepare_data_2d(test_df, feat_cols)

print("=== Data Shapes for XGBoost ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")

# =========================
# MODEL BUILD & TRAINING (Fixed Params)
# =========================
print("\n[Final] Retraining XGBoost model on full Train set with best hyperparameters...")
best_params_xgb = {
      "n_estimators": 450,
      "learning_rate": 0.0608598829841135,
      "max_depth": 3,
      "min_child_weight": 4,
      "subsample": 0.5733779454085566,
      "colsample_bytree": 0.5461692973843989,
      "gamma": 0.9313010568883545,
      "reg_alpha": 0.0011840345146135153,
      "reg_lambda": 0.00240217612024316,
      "objective": "binary:logistic",
      "eval_metric": "auc",
      "booster": "gbtree",
      "tree_method": "hist",
      "device": "cuda",
      "random_state": 1,
      "n_jobs": -1
    }

best_params_xgb.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
})

# 최종 모델은 전체 Xtr로 학습, Xva를 Early Stopping 검증용으로 사용
final_model = xgb.XGBClassifier(**best_params_xgb, early_stopping_rounds=50)

final_model.fit(
    Xtr, Ytr,
    eval_set=[(Xva, Yva)],
    verbose=False
)

# =========================
# FINAL EVALUATION
# =========================
y_prob = final_model.predict_proba(Xte)[:, 1]
y_true = Yte

# [수정] 사용된 피처 개수 추출 (XGBoost 입력 데이터의 컬럼 수)
n_features_used = Xte.shape[1]

# [수정] n_features 인자 추가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, 
    thr=OPERATING_THR,
    n_features=n_features_used  # ★ 이 부분을 추가
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    # "best_params_xgb": best_params_xgb  # (옵션) 파라미터 저장
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)


2025-12-19 00:06:48.420609: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-19 00:06:48.435979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766070408.455252  449724 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766070408.461385  449724 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766070408.476380  449724 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 


[FINAL DATA] Total Shape: (433520, 124)
[dataset(raw)] n=433,520  pos=43,352  rate=0.100000
[dataset(after 100K sampling)] n=100,000  pos=10,000  rate=0.100000
[check dataset(100K)] pos_rate=0.100000  target=0.10  diff=+0.000000  OK
=== Data Shapes for XGBoost ===
Train: X=(70000, 50), Y=(70000,)
Val  : X=(15000, 50), Y=(15000,)
Test : X=(15000, 50), Y=(15000,)

[Final] Retraining XGBoost model on full Train set with best hyperparameters...
{
  "metrics": {
    "accuracy": 0.9729999999999351,
    "precision": 0.9418695993708234,
    "recall": 0.7836601307184421,
    "specificity": 0.994506310319154,
    "f1": 0.8555119509841494,
    "roc_auc": 0.9874015847368721,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 2239.3374600152893,
    "bic": 2620.1277340195065,
    "log_likelihood": -1069.6687300076446,
    "n_features": 50,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW"
  }
}
confusion_matrix [tn fp fn tp]: 13396 74 331 1199


v4

In [2]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)
path_v0 = os.path.join(DATA_DIR, "featured_all.parquet")
final_data = pd.read_parquet(path_v0)

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")


# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=0.8, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0 + cols_v4)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_2d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).values
    Y = df[TARGET].fillna(0).astype(np.int32).values
    return X, Y

Xtr, Ytr = prepare_data_2d(train_df, feat_cols)
Xva, Yva = prepare_data_2d(val_df, feat_cols)
Xte, Yte = prepare_data_2d(test_df, feat_cols)

print("=== Data Shapes for XGBoost ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")

# =========================
# MODEL BUILD & TRAINING (Fixed Params)
# =========================
print("\n[Final] Retraining XGBoost model on full Train set with best hyperparameters...")
best_params_xgb = {
      "n_estimators": 450,
      "learning_rate": 0.0608598829841135,
      "max_depth": 3,
      "min_child_weight": 4,
      "subsample": 0.5733779454085566,
      "colsample_bytree": 0.5461692973843989,
      "gamma": 0.9313010568883545,
      "reg_alpha": 0.0011840345146135153,
      "reg_lambda": 0.00240217612024316,
      "objective": "binary:logistic",
      "eval_metric": "auc",
      "booster": "gbtree",
      "tree_method": "hist",
      "device": "cuda",
      "random_state": 1,
      "n_jobs": -1
    }

best_params_xgb.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
})

# 최종 모델은 전체 Xtr로 학습, Xva를 Early Stopping 검증용으로 사용
final_model = xgb.XGBClassifier(**best_params_xgb, early_stopping_rounds=50)

final_model.fit(
    Xtr, Ytr,
    eval_set=[(Xva, Yva)],
    verbose=False
)

# =========================
# FINAL EVALUATION
# =========================
y_prob = final_model.predict_proba(Xte)[:, 1]
y_true = Yte

# [수정] 사용된 피처 개수 추출 (XGBoost 입력 데이터의 컬럼 수)
n_features_used = Xte.shape[1]

# [수정] n_features 인자 추가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, 
    thr=OPERATING_THR,
    n_features=n_features_used  # ★ 이 부분을 추가
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    # "best_params_xgb": best_params_xgb  # (옵션) 파라미터 저장
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)



[FINAL DATA] Total Shape: (433520, 124)
[dataset(raw)] n=433,520  pos=43,352  rate=0.100000
[dataset(after 100K sampling)] n=100,000  pos=10,000  rate=0.100000
[check dataset(100K)] pos_rate=0.100000  target=0.10  diff=+0.000000  OK
=== Data Shapes for XGBoost ===
Train: X=(70000, 46), Y=(70000,)
Val  : X=(15000, 46), Y=(15000,)
Test : X=(15000, 46), Y=(15000,)

[Final] Retraining XGBoost model on full Train set with best hyperparameters...
{
  "metrics": {
    "accuracy": 0.95053333333327,
    "precision": 0.8724007561428427,
    "recall": 0.6032679738558149,
    "specificity": 0.9899777282850044,
    "f1": 0.7132921169813041,
    "roc_auc": 0.9665084841162074,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3746.326702594167,
    "bic": 4096.653754678046,
    "log_likelihood": -1827.1633512970834,
    "n_features": 46,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW"
  }
}
confusion_matrix [tn fp fn tp]: 13335 135 607 923


In [3]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb
# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."


os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)
path_v0 = os.path.join(DATA_DIR, "featured_all.parquet")
final_data = pd.read_parquet(path_v0)

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=0.8, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0 + cols_v1 + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_2d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).values
    Y = df[TARGET].fillna(0).astype(np.int32).values
    return X, Y

Xtr, Ytr = prepare_data_2d(train_df, feat_cols)
Xva, Yva = prepare_data_2d(val_df, feat_cols)
Xte, Yte = prepare_data_2d(test_df, feat_cols)

print("=== Data Shapes for XGBoost ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")

# =========================
# MODEL BUILD & TRAINING (Fixed Params)
# =========================
print("\n[Final] Retraining XGBoost model on full Train set with best hyperparameters...")
best_params_xgb = {
      "n_estimators": 450,
      "learning_rate": 0.0608598829841135,
      "max_depth": 3,
      "min_child_weight": 4,
      "subsample": 0.5733779454085566,
      "colsample_bytree": 0.5461692973843989,
      "gamma": 0.9313010568883545,
      "reg_alpha": 0.0011840345146135153,
      "reg_lambda": 0.00240217612024316,
      "objective": "binary:logistic",
      "eval_metric": "auc",
      "booster": "gbtree",
      "tree_method": "hist",
      "device": "cuda",
      "random_state": 1,
      "n_jobs": -1
    }

best_params_xgb.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
})

# 최종 모델은 전체 Xtr로 학습, Xva를 Early Stopping 검증용으로 사용
final_model = xgb.XGBClassifier(**best_params_xgb, early_stopping_rounds=50)

final_model.fit(
    Xtr, Ytr,
    eval_set=[(Xva, Yva)],
    verbose=False
)

# =========================
# FINAL EVALUATION
# =========================
y_prob = final_model.predict_proba(Xte)[:, 1]
y_true = Yte

# [수정] 사용된 피처 개수 추출 (XGBoost 입력 데이터의 컬럼 수)
n_features_used = Xte.shape[1]

# [수정] n_features 인자 추가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, 
    thr=OPERATING_THR,
    n_features=n_features_used  # ★ 이 부분을 추가
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    # "best_params_xgb": best_params_xgb  # (옵션) 파라미터 저장
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)



[FINAL DATA] Total Shape: (433520, 124)
[dataset(raw)] n=433,520  pos=43,352  rate=0.100000
[dataset(after 100K sampling)] n=100,000  pos=10,000  rate=0.100000
[check dataset(100K)] pos_rate=0.100000  target=0.10  diff=+0.000000  OK
=== Data Shapes for XGBoost ===
Train: X=(70000, 64), Y=(70000,)
Val  : X=(15000, 64), Y=(15000,)
Test : X=(15000, 64), Y=(15000,)

[Final] Retraining XGBoost model on full Train set with best hyperparameters...
{
  "metrics": {
    "accuracy": 0.9808666666666013,
    "precision": 0.962769918093103,
    "recall": 0.8450980392151339,
    "specificity": 0.9962880475129179,
    "f1": 0.900104419967903,
    "roc_auc": 0.9921535147095484,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 1761.9097254668702,
    "bic": 2249.3212761922687,
    "log_likelihood": -816.9548627334351,
    "n_features": 64,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW"
  }
}
confusion_matrix [tn fp fn tp]: 13420 50 237 1293
